<a href="https://colab.research.google.com/github/Julen10H/Neural-Network-SMS-Text-Classifier/blob/main/Copia_de_fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

train_df['label'] = train_df['label'].map({'ham': 0, 'spam': 1})
test_df['label'] = test_df['label'].map({'ham': 0, 'spam': 1})

train_df.head()

In [ ]:
from tensorflow.keras.layers import TextVectorization

MAX_TOKENS = 10000
SEQ_LEN = 100

vectorizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=SEQ_LEN
)
vectorizer.adapt(train_df['message'].values)

model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(MAX_TOKENS, 32, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    tf.constant(train_df['message'].values),
    train_df['label'].values,
    epochs=10,
    validation_data=(tf.constant(test_df['message'].values), test_df['label'].values)
)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  input_tensor = tf.constant([pred_text])
  prob = float(model.predict(input_tensor, verbose=0)[0][0])
  label = "spam" if prob > 0.5 else "ham"
  return [prob, label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


In [ ]:
test_messages = ["how are you doing today",
                 "sale today! to stop texts call 98912460324",
                 "i dont want to go. can we try it a different day? available sat",
                 "our new mobile video service is live. just install on your phone to start watching.",
                 "you have won £1000 cash! call to claim your prize.",
                 "i'll bring it tomorrow. don't forget the milk.",
                 "wow, is your arm alright. that happened to me one time too"]
test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]

for msg, ans in zip(test_messages, test_answers):
    print(ans, predict_message(msg))